# MLB Revision — Phase 2E: 반복 시드 강건성 분석 (Reviewer 1 #5)

**목적:** 원고의 DNN 결과가 특정 난수 시드(2026)에 의존하는지 확인합니다. 구조·하이퍼파라미터·분할·전처리는 **기존 그대로 고정**하고 **시드만 10개로 바꿔** 다시 학습합니다. 새로운 튜닝은 하지 않습니다.

| 과제 | 학습 자료 | 평가 | 고정 설정 |
|---|---|---|---|
| Pre-game 2025 | 2025 train (1,679경기) | 2025 9월 test (374) + **2026 전체 외부검증 (2,429)** | C1: 128-64-32, Dropout .15(1·2층 뒤), L2 1e-4, lr 1e-3, batch 64, patience 14 |
| Pre-game 2026 | 2026 train (1,682) | 2026 test (386) | 위와 동일 |
| In-game 2025 | 2025 train 스냅샷 | 2025 test (6,658) + **2026 전체 외부검증 (43,204)** | 64-32-16, Dropout .20(1·2층 뒤), L2 없음, lr 1e-3, batch 128, patience 12, 경기별 동일 가중치 |
| In-game 2026 | 2026 train 스냅샷 | 2026 test (6,872) | 위와 동일 |

- 시드: **2026–2035 (10개)**. 2026은 원고에 보고한 시드이므로, 원래 결과가 재현되는지 함께 확인합니다.
- 비교 기준: 로지스틱 회귀(결정적이므로 1회), 학습셋 홈 승률 상수 예측, In-game 경험적 승리기대표(이닝×공수×점수차 ±5, α=30).
- **해석 주의:** 2025 9월 test는 이전 단계에서 여러 번 확인했으므로, 이 분석은 *강건성 확인*이며 새 독립 검증이 아닙니다.

**업로드 파일 4개(+선택 1개):** `phase1c_output.zip`, `phase1d_output.zip`, `phase2b_2026_external_output.zip`, `phase2c_2026_retraining_output.zip` (+ 원래 C1 예측 재현 확인용 `phase1c2_output.zip`)

CPU 런타임 기준 약 15–30분 걸립니다(In-game이 대부분).

In [ ]:
import sys, subprocess, importlib.util
for pkg, mod in [('tensorflow','tensorflow'),('scikit-learn','sklearn'),('pandas','pandas'),('matplotlib','matplotlib')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import io, os, json, zipfile, random, time
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss, log_loss
print('TensorFlow', tf.__version__)

SEEDS = list(range(2026, 2036))          # 사전 지정: 10개, 2026 = 원고 시드
ORIGINAL_SEED = 2026
QUICK_TEST = False                        # True면 시드 2개·에폭 축소(코드 점검용)
if QUICK_TEST: SEEDS = SEEDS[:2]

## 1. 파일 업로드 및 자료 구성

In [ ]:
REQUIRED = ['phase1c_output.zip','phase1d_output.zip','phase2b_2026_external_output.zip','phase2c_2026_retraining_output.zip']
OPTIONAL = ['phase1c2_output.zip']
try:
    from google.colab import files
    IN_COLAB = True
    print('업로드:', REQUIRED, '(선택)', OPTIONAL)
    uploaded = files.upload()
    blobs = {k: v for k, v in uploaded.items()}
except ImportError:
    IN_COLAB = False
    base = Path(os.environ.get('PHASE2E_INPUT', '.'))
    blobs = {p.name: p.read_bytes() for p in base.glob('*.zip')}
missing = [f for f in REQUIRED if f not in blobs]
assert not missing, f'필수 파일 누락: {missing}'
zips = {k: zipfile.ZipFile(io.BytesIO(v)) for k, v in blobs.items()}
def rcsv(z, member): return pd.read_csv(io.BytesIO(zips[z].read(member)))
def rjson(z, member): return json.loads(zips[z].read(member))

PRE = ['home_win_pct','away_win_pct','win_pct_diff','home_run_diff','away_run_diff','run_diff_diff',
       'home_runs_per_game','away_runs_per_game','home_ra_per_game','away_ra_per_game',
       'home_recent_form','away_recent_form','form_diff','home_sp_era','away_sp_era','sp_era_diff',
       'home_sp_whip','away_sp_whip','sp_whip_diff','home_sp_k9','away_sp_k9','sp_k9_diff']
ING = ['inning','is_bottom','score_diff','home_runs','away_runs','home_hits','away_hits','abs_score_diff']

# --- Pre-game 2025 (Phase 1C 정제 자료: train-only 투수 중앙값, 차이변수 = 홈 - 원정)
pre25 = rcsv('phase1c_output.zip','pregame_features_cleaned_2430.csv')
audit1c = rjson('phase1c_output.zip','training_audit.json')
assert len(pre25)==2430 and pre25.split.value_counts().to_dict()=={'train':1679,'validation':377,'test':374}
for st in ['era','whip','k9']:
    assert np.allclose(pre25[f'sp_{st}_diff'], pre25[f'home_sp_{st}']-pre25[f'away_sp_{st}']), f'{st} 부호 확인'

# --- Pre-game 2026 외부검증 입력: 2025 학습셋 중앙값 + 홈-원정 차이 (Phase 2D 보정과 동일 규칙)
ext_pre = rcsv('phase2b_2026_external_output.zip','2026_pregame_features.csv')
med25 = audit1c['imputation']['pitcher_train_only_medians']
for side in ('home','away'):
    miss = ext_pre[f'{side}_sp_source'] != 'prior_game_logs'
    for st in ('era','whip','k9'):
        ext_pre.loc[miss, f'{side}_sp_{st}'] = med25[st]
for st in ('era','whip','k9'):
    ext_pre[f'sp_{st}_diff'] = ext_pre[f'home_sp_{st}'] - ext_pre[f'away_sp_{st}']
assert len(ext_pre)==2429 and ext_pre[PRE].notna().all().all()

# --- Pre-game 2026 독립 (Phase 2C 정제 자료: 2026 train-only 중앙값, 분할 포함)
pre26 = rcsv('phase2c_2026_retraining_output.zip','2026_pregame_features_training_cleaned.csv')
assert len(pre26)==2429 and set(pre26.split)=={'train','validation','test'}
for st in ['era','whip','k9']:
    assert np.allclose(pre26[f'sp_{st}_diff'], pre26[f'home_sp_{st}']-pre26[f'away_sp_{st}'])

# --- In-game 2025 / 2026
in25 = rcsv('phase1d_output.zip','ingame_snapshots_2430.csv')
assert len(in25)==43220
in26 = rcsv('phase2b_2026_external_output.zip','2026_ingame_predictions.csv').drop(columns=['prob_ingame_dnn'],errors='ignore')
part26 = rcsv('phase2c_2026_retraining_output.zip','2026_game_partitions.csv')[['gamePk','split']]
in26 = in26.merge(part26, on='gamePk', how='left', validate='many_to_one')
assert len(in26)==43204 and in26.split.notna().all()
for d in (in25, in26):
    assert (d.groupby('gamePk').split.nunique()==1).all()
print('Pre25', pre25.split.value_counts().to_dict(), '| Ext26', len(ext_pre), '| Pre26', pre26.split.value_counts().to_dict())
print('In25', in25.split.value_counts().to_dict(), '| In26', in26.split.value_counts().to_dict())

## 2. 고정 모델 구조, 지표, 비교 기준

In [ ]:
EPOCH_SCALE = 0.1 if QUICK_TEST else 1.0
def set_seed(s):
    tf.keras.backend.clear_session()
    os.environ['PYTHONHASHSEED'] = str(s); random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)
    try: tf.config.experimental.enable_op_determinism()
    except Exception: pass

def build_pre(n_in):  # Phase 1C2 C1과 동일
    reg = tf.keras.regularizers.l2(1e-4)
    inp = tf.keras.Input(shape=(n_in,))
    x = tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=reg)(inp)
    x = tf.keras.layers.Dropout(0.15)(x)
    x = tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=reg)(x)
    x = tf.keras.layers.Dropout(0.15)(x)
    x = tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=reg)(x)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    m = tf.keras.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
    return m, dict(epochs=int(180*EPOCH_SCALE) or 5, batch_size=64, patience=14)

def build_in(n_in):   # Phase 1D와 동일
    m = tf.keras.Sequential([tf.keras.Input(shape=(n_in,)),
        tf.keras.layers.Dense(64, activation='relu'), tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation='relu'), tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(16, activation='relu'), tf.keras.layers.Dense(1, activation='sigmoid')])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
    return m, dict(epochs=int(200*EPOCH_SCALE) or 5, batch_size=128, patience=12)

def ece10(y, p):
    b = np.clip(np.digitize(p, np.linspace(0,1,11)[1:-1], right=True), 0, 9)
    return float(sum((b==k).mean()*abs(y[b==k].mean()-p[b==k].mean()) for k in range(10) if (b==k).any()))
def evaluate(y, p):
    y = np.asarray(y, int); p = np.clip(np.asarray(p, float).ravel(), 1e-7, 1-1e-7)
    return dict(n=len(y), accuracy=float(accuracy_score(y, p>=.5)),
                auc=float(roc_auc_score(y, p)) if p.std()>0 else 0.5,
                brier=float(brier_score_loss(y, p)), logloss=float(log_loss(y, p, labels=[0,1])), ece10=ece10(y, p))

def game_weights(d):
    w = 1. / d.groupby('gamePk').gamePk.transform('size').to_numpy(float)
    return w / w.mean()

def ewe_fit(train):  # Phase 1D 경험적 승리기대표
    w = game_weights(train); t = train.assign(sb=train.score_diff.clip(-5,5), w=w, wy=w*train.target)
    prior = float(t.wy.sum()/t.w.sum())
    cells = t.groupby(['inning','is_bottom','sb']).agg(n=('w','sum'), wins=('wy','sum'))
    def predict(d):
        k = pd.MultiIndex.from_arrays([d.inning.astype(int), d.is_bottom.astype(int), d.score_diff.clip(-5,5).astype(int)])
        c = cells.reindex(k)
        return ((c.wins.fillna(0) + 30*prior) / (c.n.fillna(0) + 30)).to_numpy()
    return predict

TASKS = [  # name, kind, frame, features, external frame (or None)
    ('Pre-game 2025', 'pre', pre25, PRE, ext_pre),
    ('Pre-game 2026', 'pre', pre26, PRE, None),
    ('In-game 2025',  'in',  in25,  ING, in26),
    ('In-game 2026',  'in',  in26,  ING, None),
]

# 결정적 비교 기준: 로지스틱 회귀, 홈 승률 상수, (In-game) 승리기대표
ref_rows = []; ref_preds = {}
for name, kind, d, F, ext in TASKS:
    tr, te = d[d.split=='train'], d[d.split=='test']
    sc = StandardScaler().fit(tr[F].to_numpy(float))
    w = game_weights(tr) if kind=='in' else None
    lr = LogisticRegression(C=1.0, max_iter=2500, random_state=2026).fit(sc.transform(tr[F].to_numpy(float)), tr.target, sample_weight=w)
    prior = float(np.average(tr.target, weights=w))
    evals = [('test', te)] + ([('external_2026', ext)] if ext is not None else [])
    ewe = ewe_fit(tr) if kind=='in' else None
    for split, e in evals:
        preds = {'Logistic Regression': lr.predict_proba(sc.transform(e[F].to_numpy(float)))[:,1],
                 'Home-rate constant': np.full(len(e), prior)}
        if ewe: preds['Empirical Win Expectancy'] = ewe(e)
        for m, p in preds.items():
            ref_rows.append(dict(task=name, eval=split, model=m, **evaluate(e.target, p)))
            ref_preds[(name, split, m)] = p
ref = pd.DataFrame(ref_rows)
print(ref.round(4).to_string(index=False))

## 3. 시드별 재학습 (설정 고정, 검증셋 조기종료, 테스트는 학습 후 평가만)

In [ ]:
rows = []; pred_store = []
t0 = time.time()
for name, kind, d, F, ext in TASKS:
    tr, va, te = d[d.split=='train'], d[d.split=='validation'], d[d.split=='test']
    sc = StandardScaler().fit(tr[F].to_numpy(float))
    X = lambda frame: sc.transform(frame[F].to_numpy(float)).astype('float32')
    Xtr, Xva = X(tr), X(va)
    w = game_weights(tr) if kind=='in' else None
    evals = [('test', te)] + ([('external_2026', ext)] if ext is not None else [])
    for s in SEEDS:
        set_seed(s)
        model, cfg = (build_pre if kind=='pre' else build_in)(len(F))
        h = model.fit(Xtr, tr.target.to_numpy(int), sample_weight=w,
                      validation_data=(Xva, va.target.to_numpy(int)),
                      epochs=cfg['epochs'], batch_size=cfg['batch_size'], verbose=0,
                      callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=cfg['patience'], restore_best_weights=True)])
        best = int(np.argmin(h.history['val_loss']) + 1)
        val_m = evaluate(va.target, model.predict(Xva, verbose=0))
        for split, e in evals:
            p = model.predict(X(e), verbose=0).ravel()
            rows.append(dict(task=name, eval=split, seed=s, best_epoch=best, epochs_run=len(h.history['loss']),
                             val_brier=val_m['brier'], val_auc=val_m['auc'], **evaluate(e.target, p)))
            keys = ['gamePk'] + (['inning','is_bottom'] if kind=='in' else [])
            pred_store.append(e[keys+['target']].assign(task=name, eval=split, seed=s, prob=p))
        r = rows[-len(evals)]
        print(f"{name:14s} seed {s}  best_epoch {best:3d}  test acc {r['accuracy']:.4f} AUC {r['auc']:.4f} Brier {r['brier']:.4f}  ({time.time()-t0:.0f}s)")
per_seed = pd.DataFrame(rows)
preds = pd.concat(pred_store, ignore_index=True)

## 4. 요약: 평균±SD, 범위, 원고 시드 위치, 비교 기준 대비

In [ ]:
summ = []
for (task, ev), g in per_seed.groupby(['task','eval'], sort=False):
    row = dict(task=task, eval=ev, n_seeds=len(g))
    for m in ['accuracy','auc','brier','logloss','ece10','best_epoch']:
        row[f'{m}_mean'] = g[m].mean(); row[f'{m}_sd'] = g[m].std(ddof=1)
        row[f'{m}_min'] = g[m].min(); row[f'{m}_max'] = g[m].max()
    o = g[g.seed==ORIGINAL_SEED]
    if len(o):
        row['orig_seed_auc'] = float(o.auc.iloc[0]); row['orig_seed_brier'] = float(o.brier.iloc[0])
        row['orig_seed_auc_rank'] = int((g.auc > o.auc.iloc[0]).sum() + 1)
    lr = ref[(ref.task==task)&(ref['eval']==ev)&(ref.model=='Logistic Regression')].iloc[0]
    hc = ref[(ref.task==task)&(ref['eval']==ev)&(ref.model=='Home-rate constant')].iloc[0]
    row.update(lr_auc=lr.auc, lr_brier=lr.brier, homeconst_brier=hc.brier, homeconst_acc=hc.accuracy,
               seeds_brier_below_lr=int((g.brier < lr.brier).sum()),
               seeds_brier_below_homeconst=int((g.brier < hc.brier).sum()))
    e = ref[(ref.task==task)&(ref['eval']==ev)&(ref.model=='Empirical Win Expectancy')]
    if len(e): row.update(ewe_brier=float(e.brier.iloc[0]), seeds_brier_below_ewe=int((g.brier < e.brier.iloc[0]).sum()))
    summ.append(row)
summary = pd.DataFrame(summ)
show = summary[['task','eval','n_seeds','accuracy_mean','accuracy_sd','auc_mean','auc_sd','auc_min','auc_max',
                'brier_mean','brier_sd','best_epoch_min','best_epoch_max','orig_seed_auc','orig_seed_auc_rank',
                'lr_auc','lr_brier','homeconst_brier','seeds_brier_below_lr','seeds_brier_below_homeconst']]
print(show.round(4).to_string(index=False))

# 원고 시드(2026) 재현 확인: Phase 1C2 저장 예측과 비교 (선택 파일이 있을 때)
repro = {}
if 'phase1c2_output.zip' in zips:
    old = rcsv('phase1c2_output.zip','selected_test_predictions.csv')[['gamePk','prob_selected_dnn']]
    new = preds[(preds.task=='Pre-game 2025')&(preds['eval']=='test')&(preds.seed==ORIGINAL_SEED)][['gamePk','prob']]
    mm = old.merge(new, on='gamePk', validate='one_to_one')
    repro = dict(max_abs_diff=float(np.abs(mm.prob - mm.prob_selected_dnn).max()), n=len(mm))
    print('원고 시드 2026 재현 (Pre-game 2025 test) 최대 절대 차이:', repro['max_abs_diff'],
          '→ 0에 가까우면 재현, 크면 하드웨어/버전 차이로 학습 경로가 달라진 것(원고에는 이 노트북의 10시드 결과를 보고)')

## 5. 그림 및 결과 ZIP

In [ ]:
evals = list(summary[['task','eval']].itertuples(index=False, name=None))
fig, axes = plt.subplots(2, len(evals), figsize=(3.2*len(evals), 6), squeeze=False)
for j, (task, ev) in enumerate(evals):
    g = per_seed[(per_seed.task==task)&(per_seed['eval']==ev)]
    for i, metric in enumerate(['auc','brier']):
        ax = axes[i][j]
        ax.scatter(np.zeros(len(g)) + np.random.default_rng(0).uniform(-.08,.08,len(g)), g[metric], s=22, label='DNN seeds')
        o = g[g.seed==ORIGINAL_SEED]
        if len(o): ax.scatter([0], o[metric], s=70, facecolors='none', edgecolors='k', label='seed 2026')
        for m, ls in [('Logistic Regression','--'),('Home-rate constant',':'),('Empirical Win Expectancy','-.')]:
            r = ref[(ref.task==task)&(ref['eval']==ev)&(ref.model==m)]
            if len(r) and not (metric=='auc' and m=='Home-rate constant'):
                ax.axhline(float(r[metric].iloc[0]), ls=ls, lw=1, color='gray', label=m)
        ax.set_xticks([]); ax.set_title(f'{task}\n{ev}' if i==0 else '', fontsize=9); ax.set_ylabel(metric.upper() if j==0 else '')
axes[0][0].legend(fontsize=6, loc='best')
plt.tight_layout(); plt.savefig('phase2e_seed_robustness.png', dpi=160); plt.show()

out = Path('phase2e_seed_robustness_output'); out.mkdir(exist_ok=True)
per_seed.to_csv(out/'per_seed_metrics.csv', index=False)
summary.to_csv(out/'seed_summary.csv', index=False)
ref.to_csv(out/'deterministic_reference_models.csv', index=False)
preds.to_csv(out/'per_seed_predictions.csv.gz', index=False, compression='gzip')
Path('phase2e_seed_robustness.png').replace(out/'phase2e_seed_robustness.png')
audit = dict(purpose='Reviewer 1 #5 repeated-seed robustness; architecture/hyperparameters/splits fixed, no new tuning',
             seeds=SEEDS, original_seed=ORIGINAL_SEED, quick_test=QUICK_TEST, tensorflow=tf.__version__,
             pre_config='22-128-64-32-1, dropout .15 after hidden 1-2, L2 1e-4 hidden, Adam 1e-3, batch 64, patience 14, max 180 epochs',
             in_config='8-64-32-16-1, dropout .20 after hidden 1-2, no L2, Adam 1e-3, batch 128, patience 12, max 200 epochs, equal per-game weights',
             external_2026_pregame_input='2025 train-only pitcher medians for non-prior-log starters; sp_*_diff = home - away (Phase 2D rule)',
             original_seed_reproduction=repro,
             caveats=['2025 Sep test was inspected in earlier phases: robustness check, not a new holdout',
                      '2026 external = retrospective reconstruction; probable-pitcher pre-announcement unverified',
                      'op determinism and results may differ slightly across hardware/TensorFlow versions'])
(out/'phase2e_audit.json').write_text(json.dumps(audit, indent=2, ensure_ascii=False))
with zipfile.ZipFile('phase2e_seed_robustness_output.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in out.iterdir(): z.write(p, arcname=p.name)
print('완료: phase2e_seed_robustness_output.zip')
if IN_COLAB: files.download('phase2e_seed_robustness_output.zip')